<a href="https://colab.research.google.com/github/kyati001/DECODING-CUSTOMER-VALUE/blob/main/SqL_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Section

In [31]:
import pandas as pd
import numpy as np

def engineer_customer_features(file_path):
    try:
        df = pd.read_csv('/content/Dataset.csv')
    except FileNotFoundError:
        return f"❌ Error: Could not find '{file_path}'. Please check the filename in your sidebar."

    # Clean and standardize column names explicitly
    df.columns = df.columns.str.strip()

    mapping = {
        'Purchase Amount (USD)': 'Purchase_Amount_USD',
        'Review Rating': 'Review_Rating',
        'Promo Code Used': 'Promo_Code_Used',
        'Discount Applied': 'Discount_Applied',
        'Previous Purchases': 'Previous_Purchases',
        'Frequency of Purchases': 'Frequency_of_Purchases',
        'Item Purchased': 'Item_Purchased',
        'Location': 'Location'
    }
    df = df.rename(columns=mapping)

    # Cast numerical columns safely
    df['Purchase_Amount_USD'] = pd.to_numeric(df['Purchase_Amount_USD'], errors='coerce')
    df['Review_Rating'] = pd.to_numeric(df['Review_Rating'], errors='coerce')
    df['Previous_Purchases'] = pd.to_numeric(df['Previous_Purchases'], errors='coerce').fillna(0)

    # Process promotional flags
    df['Promo_Used_Numeric'] = df['Promo_Code_Used'].apply(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)

    # Engineer baseline profiles
    customer_profile = df.copy()
    customer_profile['Promo_Reliance_Ratio'] = customer_profile['Promo_Used_Numeric']
    customer_profile['Estimated_Total_Value'] = customer_profile['Purchase_Amount_USD'] * (customer_profile['Previous_Purchases'] + 1)
    customer_profile['Satisfaction_Flag'] = customer_profile['Review_Rating'].apply(lambda x: 1 if x >= 4.0 else 0)

    output_path = "cleaned_customer_intelligence.csv"
    customer_profile.to_csv(output_path, index=False)
    print(f"🎉 Success! Fashion dataset processed and saved to: {output_path}\n")

    return customer_profile

# 🚀 EXECUTION LINE: Change "fashion_trends.csv" to your actual file name!
df_engineered = engineer_customer_features("fashion_trends.csv")

if isinstance(df_engineered, pd.DataFrame):
    display(df_engineered[['Item_Purchased', 'Location', 'Promo_Reliance_Ratio', 'Estimated_Total_Value', 'Satisfaction_Flag']].head())

🎉 Success! Fashion dataset processed and saved to: cleaned_customer_intelligence.csv



,Item_Purchased,Location,Promo_Reliance_Ratio,Estimated_Total_Value,Satisfaction_Flag
0,Blouse,Kentucky,1,795,0
1,Sweater,Maine,1,192,0
2,Jeans,Massachusetts,1,1752,0
3,Sandals,Rhode Island,1,4500,0
4,Blouse,Oregon,1,1568,0


In [34]:
# Definition A: Volume/Tenure Focus
df_engineered['Loyalty_Def_A'] = np.where(
    (df_engineered['Previous_Purchases'] > 20) &
    (df_engineered['Frequency_of_Purchases'].str.strip().isin(['Weekly', 'Bi-Weekly', 'Fortnightly'])),
    1, 0
)

# Definition B: Margin/Organic Focus
median_spend = df_engineered['Purchase_Amount_USD'].median()
df_engineered['Loyalty_Def_B'] = np.where(
    (df_engineered['Purchase_Amount_USD'] > median_spend) &
    (df_engineered['Promo_Code_Used'].str.strip().str.lower() == 'no'),
    1, 0
)

# Statistical validation against financial performance
corr_A = df_engineered['Loyalty_Def_A'].corr(df_engineered['Estimated_Total_Value'])
corr_B = df_engineered['Loyalty_Def_B'].corr(df_engineered['Estimated_Total_Value'])

print("--- LOYALTY VALIDATION MATRIX ---")
print(f"📊 Definition A (Frequency/Tenure) Correlation with Total Value: {corr_A:.2f}")
print(f"📊 Definition B (Organic Value/Margin) Correlation with Total Value: {corr_B:.2f}")

--- LOYALTY VALIDATION MATRIX ---
📊 Definition A (Frequency/Tenure) Correlation with Total Value: 0.31
📊 Definition B (Organic Value/Margin) Correlation with Total Value: 0.31


In [35]:
!pip install pandasql

  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=775162c5b849921e403a8cbcf2d22567bfa1596ac6ca5546426bb513ef3e19f1
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql


In [36]:
from pandasql import sqldf
pysqldf = lambda q: sqldf(q, globals())

query_pyramid = """
SELECT
    Location,
    Category,
    COUNT(*) as Customer_Count,
    AVG(Purchase_Amount_USD) as Avg_Order_Value,
    AVG(Estimated_Total_Value) as Avg_Lifetime_Value,
    CASE
        WHEN Estimated_Total_Value >= 3000 AND Promo_Reliance_Ratio = 0 THEN 'Platinum (Organic High-Value)'
        WHEN Estimated_Total_Value >= 1500 THEN 'Gold (High-Value)'
        WHEN Estimated_Total_Value >= 500 THEN 'Silver (Mid-Value)'
        ELSE 'Bronze (Low-Value/Opportunistic)'
    END as Value_Tier
FROM df_engineered
GROUP BY Value_Tier, Location, Category
ORDER BY Avg_Lifetime_Value DESC;
"""

df_pyramid = pysqldf(query_pyramid)
display(df_pyramid.head(10))

,Location,Category,Customer_Count,Avg_Order_Value,Avg_Lifetime_Value,Value_Tier
0,Louisiana,Outerwear,1,99.0,5049.0,Gold (High-Value)
1,Arizona,Footwear,1,100.0,5000.0,Platinum (Organic High-Value)
2,Idaho,Outerwear,1,99.0,4950.0,Platinum (Organic High-Value)
3,Arizona,Outerwear,1,100.0,4900.0,Platinum (Organic High-Value)
4,Oregon,Accessories,1,96.0,4896.0,Platinum (Organic High-Value)
5,Utah,Footwear,1,96.0,4800.0,Platinum (Organic High-Value)
6,Kentucky,Outerwear,1,95.0,4750.0,Platinum (Organic High-Value)
7,Nevada,Footwear,1,98.0,4606.0,Platinum (Organic High-Value)
8,New York,Accessories,1,98.0,4606.0,Platinum (Organic High-Value)
9,Texas,Accessories,1,94.0,4606.0,Platinum (Organic High-Value)


In [37]:
query_sensitivity = """
SELECT
    CASE
        WHEN Promo_Reliance_Ratio = 1 AND Previous_Purchases <= 10 THEN 'Bargain Hunter (At-Risk)'
        WHEN Promo_Reliance_Ratio = 1 AND Previous_Purchases > 10 THEN 'Discount-Dependent Loyalist'
        WHEN Promo_Reliance_Ratio = 0 AND Previous_Purchases > 10 THEN 'Organic Brand Advocate'
        ELSE 'Core Regular Customer'
    END as Behavioral_Segment,
    COUNT(*) as Total_Customers,
    SUM(Purchase_Amount_USD) as Total_Direct_Revenue,
    AVG(Review_Rating) as Avg_Satisfaction
FROM df_engineered
GROUP BY Behavioral_Segment;
"""

df_sensitivity = pysqldf(query_sensitivity)
display(df_sensitivity)

,Behavioral_Segment,Total_Customers,Total_Direct_Revenue,Avg_Satisfaction
0,Bargain Hunter (At-Risk),317,18757,3.790675
1,Core Regular Customer,467,28807,3.769165
2,Discount-Dependent Loyalist,1360,80654,3.727765
3,Organic Brand Advocate,1756,104863,3.754670


In [38]:
query_geography = """
SELECT
    Location as State,
    COUNT(*) as Customer_Base,
    SUM(Purchase_Amount_USD) as Total_Revenue,
    AVG(Promo_Reliance_Ratio) as Average_Promo_Dependency,
    AVG(Purchase_Amount_USD) as Basket_Size,
    SUM(CASE WHEN Promo_Reliance_Ratio = 0 THEN Purchase_Amount_USD ELSE 0 END) * 100.0 / SUM(Purchase_Amount_USD) as Organic_Revenue_Share_Percent
FROM df_engineered
GROUP BY Location
HAVING Customer_Base > 10
ORDER BY Organic_Revenue_Share_Percent DESC, Total_Revenue DESC;
"""

df_geography = pysqldf(query_geography)
display(df_geography.head(10))

,State,Customer_Base,Total_Revenue,Average_Promo_Dependency,Basket_Size,Organic_Revenue_Share_Percent
0,Kansas,63,3437,0.238095,54.555556,77.800407
1,Connecticut,78,4226,0.333333,54.179487,70.018930
2,Tennessee,77,4772,0.363636,61.974026,66.701593
3,Maine,77,4388,0.350649,56.987013,65.884230
4,Arizona,65,4326,0.338462,66.553846,65.788257
5,Texas,77,4712,0.363636,61.194805,63.752122
6,Alaska,72,4867,0.402778,67.597222,63.180604
7,Michigan,73,4533,0.397260,62.095890,63.136995
8,Virginia,77,4842,0.376623,62.883117,62.722016
9,Montana,96,5784,0.375000,60.250000,62.551867


In [39]:
query_funnel = """
SELECT
    Category,
    Item_Purchased,
    COUNT(*) as Transaction_Volume,
    AVG(Previous_Purchases) as Avg_Customer_Tenure,
    SUM(Purchase_Amount_USD) as Category_Revenue,
    AVG(Review_Rating) as Product_Satisfaction
FROM df_engineered
GROUP BY Category, Item_Purchased
ORDER BY Avg_Customer_Tenure DESC;
"""

df_funnel = pysqldf(query_funnel)
display(df_funnel)

,Category,Item_Purchased,Transaction_Volume,Avg_Customer_Tenure,Category_Revenue,Product_Satisfaction
0,Accessories,Jewelry,171,28.906433,10010,3.758824
1,Outerwear,Coat,161,26.813665,9275,3.727673
2,Accessories,Scarf,157,26.783439,9561,3.705806
3,Clothing,Blouse,171,26.684211,10410,3.680473
4,Clothing,Dress,166,26.548193,10320,3.750000
5,Accessories,Gloves,140,26.442857,8477,3.862774
6,Footwear,Boots,144,26.361111,9018,3.818881
7,Clothing,Shorts,157,26.038217,9433,3.714194
8,Clothing,Shirt,169,25.668639,10332,3.622024
9,Clothing,Sweater,164,25.603659,9462,3.760870


In [40]:
df_engineered.to_csv("D2C_Fashion_Final_Cleaned.csv", index=False)